<div dir="rtl">

# جيم بوت — تطبيق مقرر M269 (الخوارزميات وهياكل البيانات والحوسبة)

الدفتر يقيس كل خوارزمية في جيم بوت ويقارنها بالنظرية، ويبيّن كيف غيّر هذا التحليل الروبوت نفسه:

1. **التعقيد**: هل الزمن المقاس يطابق رتبة النمو النظرية؟
2. **هياكل البيانات**: الطابور، والكومة، وجدول التجزئة، من الصفر
3. **الجشع مقابل البرمجة الديناميكية**: ترتيب جمع العملات، وخطة المستشار
4. **الرسوم البيانية**: BFS و Dijkstra و A* على خريطة بتضاريس
5. **الاستدعاء الذاتي والذاكرة**: Minimax في «إكس أو»
6. **حدود الحوسبة**: NP-hard، والزمن شبه كثير الحدود، ومسألة التوقف

كل التجارب بذور ثابتة، فتعطي نفس الأرقام عند إعادة التشغيل (عدا الأزمنة بالملي ثانية).
</div>

In [1]:
import sys, json, math
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "algorithms").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from algorithms import experiments as E, figures as F, structures as S, sorting, graphs as G, tsp, knapsack
FIG = ROOT / "algorithms" / "figures"; FIG.mkdir(exist_ok=True)
R = {}

<div dir="rtl">

## 1. التعقيد: النظرية مقابل القياس

إذا كان الزمن ≈ c·nᵏ فإن log(الزمن) = k·log(n) + ثابت، أي أن **ميل** الخط على محورين لوغاريتميين يقدّر k.
ميل ≈ 1 يعني خطياً (أو n log n)، وميل ≈ 2 يعني تربيعياً.

### 1.1 ترتيب لوحة المتصدرين
</div>

In [2]:
R["sorting"] = E.sorting_growth()
F.sorting_loglog(R["sorting"], "ar", FIG / "sorting_ar.png")
pd.DataFrame({k: {"ميل الزمن": v["slope"], "ميل المقارنات": v["slope_comparisons"],
                  "مقارنات عند n=2000": v["comparisons"][v["n"].index(2000)]} for k, v in R["sorting"].items()}).T

,ميل الزمن,ميل المقارنات,مقارنات عند n=2000
insertion,2.05,2.01,1018007.0
merge,1.14,1.17,19379.0
quick,1.14,1.15,25259.0
heap,1.15,1.18,37772.0


<div dir="rtl">

**ميل المقارنات** هو المقياس الثابت: الإدراج 2.0 (تربيعي)، والبقية نحو 1.15 (n log n يبدو أكبر قليلاً من 1 على
هذه الأحجام). ميل **الزمن** يتغير قليلاً بين تشغيل وآخر لأنه يتأثر بالذاكرة المؤقتة وبمفسّر بايثون.
عند 2,000 لاعب يحتاج الإدراج نحو مليون مقارنة، والدمج نحو 19 ألفاً فقط.

### 1.2 البحث: خطي مقابل ثنائي (أسوأ حالة)
</div>

In [3]:
R["search"] = E.search_growth(); pd.DataFrame(R["search"])

,n,linear,binary,log2n
0,1000,1000,10,10.0
1,10000,10000,14,13.3
2,100000,100000,17,16.6
3,1000000,1000000,20,19.9


<div dir="rtl">

### 1.3 BFS على خرائط أكبر
</div>

In [4]:
R["bfs"] = E.bfs_growth(); print("ميل الزمن مقابل عدد الخلايا V:", R["bfs"]["slope"]); pd.DataFrame(R["bfs"]).drop(columns="slope")

ميل الزمن مقابل عدد الخلايا V: 0.98


,V,ms
0,90,0.185791
1,360,0.547219
2,1440,2.326413
3,5760,9.593509
4,23040,39.618826


<div dir="rtl">

## 2. هياكل البيانات

### 2.1 الكومة: heapify مقابل n عمليات push
</div>

In [5]:
R["heap"] = E.heap_build(); pd.DataFrame(R["heap"])

,n,heapify,pushes,heapify_per_n,push_per_n
0,1000,1875,2238,1.88,2.24
1,10000,18832,22772,1.88,2.28
2,100000,188081,228401,1.88,2.28


<div dir="rtl">

البناء من الأسفل للأعلى يحتاج نحو 1.9n مقارنة مهما كبر n (أي O(n))، والإدخال واحداً واحداً أكثر.
وفي ترتيب الكومة نحتاج بعد ذلك n عمليات pop، كل واحدة O(log n).

### 2.2 جدول التجزئة
</div>

In [6]:
R["hashmap"] = E.hashmap_probes(); pd.Series(R["hashmap"])

n                     50000.00
avg_probes_per_get        1.19
load                      0.38
longest_chain             5.00
resizes                  14.00
dtype: float64

<div dir="rtl">

متوسط 1.2 مقارنة لكل بحث بين 50 ألف مفتاح: هذا معنى O(1) في المتوسط، لأن التكبير يُبقي الحمل تحت 0.75.
</div>

<div dir="rtl">

## 3. الجشع مقابل البرمجة الديناميكية

### 3.1 ترتيب جمع العملات
كان الروبوت يذهب دائماً لأقرب عملة. نقارن ذلك بالترتيب الأمثل (Held-Karp) على 3,000 خريطة من محرك اللعبة نفسه:
</div>

In [7]:
R["routes"] = E.coin_routes()
F.gap_hist(R["routes"], "ar", FIG / "gap_ar.png")
pd.Series({k: v for k, v in R["routes"].items() if k != "hist"})

maps                  3000.00
worse_pct               44.70
avg_gap                  1.26
max_gap                 10.00
avg_gap_when_worse       2.81
dtype: float64

In [8]:
R["route_steps"] = E.coin_route_steps(); pd.Series(R["route_steps"])

maps          3000.00
old_avg         21.40
new_avg         20.07
saving_pct       6.20
dtype: float64

<div dir="rtl">

«الأقرب أولاً» ليس الأقصر في نحو 45% من الخرائط، وقد يزيد حتى 10 خطوات. **لذلك صار الروبوت يستخدم Held-Karp**،
فقلّ متوسط الخطوات الفعلية من 21.4 إلى 20.1 (6.2%). والحساب سريع: 6 عملات = 186 حالة فقط.

### 3.2 لماذا لا نجرّب كل الترتيبات؟
</div>

In [9]:
R["tsp"] = E.tsp_growth()
F.tsp_growth(R["tsp"], "ar", FIG / "tsp_ar.png")
pd.DataFrame(R["tsp"]["held_karp"]).merge(pd.DataFrame(R["tsp"]["brute_force"]), on="k", how="left",
                                         suffixes=("_held_karp", "_brute"))

,k,ms_held_karp,states,ms_brute,orders,tried
0,4,0.045453,28,0.026023,24.0,24.0
1,5,0.070396,75,0.037022,120.0,120.0
2,6,0.193424,186,0.202550,720.0,720.0
3,7,0.432296,441,1.534243,5040.0,5040.0
4,8,1.004201,1016,13.686906,40320.0,40320.0
5,9,2.805442,2295,138.613858,362880.0,362880.0
6,10,6.874684,5110,NaN,NaN,NaN
7,11,15.164355,11253,NaN,NaN,NaN
8,12,36.897582,24564,NaN,NaN,NaN
9,13,166.989753,53235,NaN,NaN,NaN


<div dir="rtl">

البحث الشامل ينمو مثل k! والبرمجة الديناميكية مثل 2ᵏ·k². الاثنان أُسّيان، لكن الفرق ضخم:
عند 14 عملة، 14! ≈ 87 مليار ترتيب، مقابل نحو 115 ألف حالة.

### 3.3 خطة المستشار (حقيبة الظهر 0/1)
</div>

In [10]:
R["knapsack"] = E.knapsack_compare(); pd.Series(R["knapsack"])

instances              2000.00
greedy_worse_pct          7.00
greedy_avg_loss_pct       0.47
avg_dp_cells            162.00
avg_backtrack_nodes      21.00
dtype: float64

In [11]:
R["knapsack_T"] = E.knapsack_pseudo_poly(); print("ميل زمن DP مقابل T:", R["knapsack_T"]["slope"]); pd.DataFrame(R["knapsack_T"]).drop(columns=["slope","n"])

ميل زمن DP مقابل T: 1.16


,T,ms
0,60,0.202639
1,120,0.455948
2,240,0.936748
3,480,2.358771
4,960,5.360847
5,1920,10.526323


<div dir="rtl">

الجشع يخطئ في 7% من الحالات المشابهة لمهام جيم بوت، لذلك يستخدم المستشار البرمجة الديناميكية، واختبار تلقائي
يقارنه بالبحث الشامل على 150 حالة. وزمن DP يتضاعف تقريباً عندما تتضاعف الدقائق T: هذا «شبه كثير حدود».
</div>

<div dir="rtl">

## 4. الرسوم البيانية مع التضاريس

خرائط 12×12 فيها جدران وطين (تكلفة 3). BFS يجد أقل **عدد** خطوات، لا أقل **تكلفة**:
</div>

In [12]:
R["terrain"] = E.terrain_search()
F.search_nodes(R["terrain"], "ar", FIG / "search_ar.png")
pd.Series(R["terrain"])

maps                 500.00
bfs_worse_pct         53.60
bfs_extra_cost         2.04
expanded_bfs          64.60
expanded_dijkstra     64.90
expanded_astar        26.20
astar_saving_pct      59.60
dtype: float64

<div dir="rtl">

مسار BFS أغلى في أكثر من نصف الخرائط. و A* يجد نفس تكلفة Dijkstra (لأن مسافة مانهاتن لا تبالغ أبداً)
لكنه يفحص عقداً أقل بنحو 60%.
</div>

<div dir="rtl">

## 5. الاستدعاء الذاتي والذاكرة: Minimax
</div>

In [13]:
R["minimax"] = E.minimax_nodes(); pd.Series(R["minimax"])

value_from_empty                0.0
calls_without_memo         549946.0
unique_states_with_memo      5478.0
cache_hits                  10690.0
reduction                     100.4
dtype: float64

<div dir="rtl">

Minimax بالاستدعاء الذاتي يزور 549,946 عقدة من لوحة فارغة، لكن عدد الوضعيات المختلفة 5,478 فقط.
الذاكرة (memoisation بجدول تجزئة، `lru_cache` في المحرك) تقلل العمل نحو 100 مرة. والقيمة 0 تعني أن اللعب المثالي
من الطرفين ينتهي بالتعادل، ولهذا لا يخسر الروبوت أبداً.
</div>

<div dir="rtl">

## 6. حدود الحوسبة

* **ترتيب جمع العملات** نسخة من مسألة البائع المتجول (TSP)، وهي NP-hard: لا نعرف خوارزمية زمنها كثير حدود في k.
  نحلها بدقة لأن k = 6 صغير، ولو كبرت الخريطة لاحتجنا تقريباً (مثل الأقرب أولاً مع تحسين 2-opt).
* **خطة المستشار** حقيبة ظهر 0/1، ونسختها القرارية NP-complete. الحل O(n·T) ليس كثير حدود في **حجم** المدخلات،
  لأن T يُكتب بـ log T بت. لكنه سريع عملياً لأن T ≤ 240 دقيقة.
* **مسألة التوقف**: لو سمح التطبيق للمستخدم بكتابة برنامج لروبوته، لا يمكن لأي برنامج أن يقرر مسبقاً لكل البرامج
  هل ستنتهي (تورنغ 1936). لذلك المهام في جيم بوت **بيانات** (مكافأة ومدة)، وأي تنفيذ يجب أن تكون له مهلة.
</div>

In [14]:
(FIG / "results.json").write_text(json.dumps(R, ensure_ascii=False, indent=1, default=float))
print("محفوظ:", sorted(p.name for p in FIG.glob("*")))

محفوظ: ['gap_ar.png', 'results.json', 'search_ar.png', 'sorting_ar.png', 'tsp_ar.png']
